In [22]:
# notebook designed to see whether the ROOT-based TMVA reader ane my own evaluator give the same output MVA score
from pathlib import Path
import sys
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# import ROOT 
# -> issues with ROOT, so try on SWAN for the part testing the ROOT-based TMVA reader

In [16]:
# Change this if the notebook is not run from the repository root.

REPO = Path.cwd()

# If notebook is launched from analysis/tthMVA
if REPO.name == "tthMVA":
    TTHMVA_DIR = REPO

# If notebook is launched from higgscharm
elif REPO.name in ["higgscharm", "ua-cms-higgscharm"]:
    TTHMVA_DIR = REPO / "analysis" / "tthMVA"

else:
    raise RuntimeError(
        "Please run this notebook either from "
        "the repository root or from analysis/tthMVA."
    )

REPO = TTHMVA_DIR.parent.parent

print("Repository:")
print(REPO)

print("\nTTHMVA directory:")
print(TTHMVA_DIR)

Repository:
/home/cms-jovyan/ua-cms-higgscharm

TTHMVA directory:
/home/cms-jovyan/ua-cms-higgscharm/analysis/tthMVA


In [19]:
JSON_FILE = (
    REPO
    / "analysis/data/tthMVA_2022-2023_retrained"
    / "Muon-mvaTTH.2022EE.weights.json"
)

XML_FILE = (
    REPO
    / "analysis/data/tthMVA_2022-2023_retrained"
    / "Muon-mvaTTH.2022EE.weights.xml"
)

print("JSON:")
print(JSON_FILE)

print("\nXML:")
print(XML_FILE)

assert JSON_FILE.exists(), f"JSON file does not exist: {JSON_FILE}"
assert XML_FILE.exists(), f"XML file does not exist: {XML_FILE}"

JSON:
/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/Muon-mvaTTH.2022EE.weights.json

XML:
/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/Muon-mvaTTH.2022EE.weights.xml


In [20]:
sys.path.insert(0, str(TTHMVA_DIR))

from evaluator import TMVAGradBDT

print("Imported:")
print(TMVAGradBDT)

Imported:
<class 'evaluator.TMVAGradBDT'>


In [23]:
# inspect json file
with open(JSON_FILE, "r") as f:
    model = json.load(f)

print("Top-level keys:")
print(model.keys())

Top-level keys:
dict_keys(['format', 'method', 'ntrees', 'variables', 'options', 'trees'])


In [24]:
# inspect json content
print("Format:", model["format"])
print("Method:", model["method"])
print("Number of trees:", model["ntrees"])
print("Number of variables:", len(model["variables"]))
print("BoostType:", model["options"].get("BoostType"))

Format: TMVA_BDT_JSON
Method: BDT::BDTG
Number of trees: 500
Number of variables: 13
BoostType: Grad


In [25]:
assert model["format"] == "TMVA_BDT_JSON"

assert model["ntrees"] == 500

assert len(model["variables"]) == 13

assert model["options"]["BoostType"] == "Grad"

assert len(model["trees"]) == 500

print("Basic JSON validation: PASSED")

Basic JSON validation: PASSED


In [26]:
# verify variables
for variable in model["variables"]:
    print(
        f'{variable["index"]:2d}  '
        f'{variable["label"]:35s}  '
        f'{variable["expression"]}'
    )

 0  Muon_pt                              Muon_pt
 1  Muon_eta                             Muon_eta
 2  Muon_pfRelIso03_all                  Muon_pfRelIso03_all
 3  Muon_miniPFRelIso_chg                Muon_miniPFRelIso_chg
 4  Muon_miniRelIsoNeutral               Muon_miniPFRelIso_all-Muon_miniPFRelIso_chg
 5  Muon_jetNDauCharged                  Muon_jetNDauCharged
 6  Muon_jetPtRelv2                      Muon_jetPtRelv2
 7  Muon_jetBTagDeepFlavB                Muon_jetIdx>-1?Jet_btagDeepFlavB[Muon_jetIdx]:0
 8  Muon_jetPtRatio                      min(1/(1+Muon_jetRelIso),1.5)
 9  Muon_sip3d                           Muon_sip3d
10  Muon_log_dxy                         log(abs(Muon_dxy))
11  Muon_log_dz                          log(abs(Muon_dz))
12  Muon_segmentComp                     Muon_segmentComp


In [28]:
# verify variables
expected_variables = [
    "Muon_pt",
    "Muon_eta",
    "Muon_pfRelIso03_all",
    "Muon_miniPFRelIso_chg",
    "Muon_miniRelIsoNeutral",
    "Muon_jetNDauCharged",
    "Muon_jetPtRelv2",
    "Muon_jetBTagDeepFlavB",
    "Muon_jetPtRatio",
    "Muon_sip3d",
    "Muon_log_dxy",
    "Muon_log_dz",
    "Muon_segmentComp",
]

json_variables = [
    variable["label"]
    for variable in model["variables"]
]

assert json_variables == expected_variables

print("Variable ordering: PASSED")

Variable ordering: PASSED


In [29]:
# create ROOT-free evaluator
python_evaluator = TMVAGradBDT(JSON_FILE)

print("TMVAGradBDT created successfully.")
print("Number of trees:", python_evaluator.ntrees)
print("Variables:", python_evaluator.variable_names)

TMVAGradBDT created successfully.
Number of trees: 500
Variables: ['Muon_pt', 'Muon_eta', 'Muon_pfRelIso03_all', 'Muon_miniPFRelIso_chg', 'Muon_miniRelIsoNeutral', 'Muon_jetNDauCharged', 'Muon_jetPtRelv2', 'Muon_jetBTagDeepFlavB', 'Muon_jetPtRatio', 'Muon_sip3d', 'Muon_log_dxy', 'Muon_log_dz', 'Muon_segmentComp']


In [30]:
# assert that it has the correct model
assert python_evaluator.ntrees == 500

assert python_evaluator.variable_names == expected_variables

assert len(python_evaluator.trees) == 500

print("Evaluator internal structure: PASSED")

Evaluator internal structure: PASSED


In [31]:
# test on a single artificial event
test_event = np.array([
    50.0,     # Muon_pt
    0.5,      # Muon_eta
    0.05,     # Muon_pfRelIso03_all
    0.02,     # Muon_miniPFRelIso_chg
    0.03,     # neutral mini isolation
    5.0,      # Muon_jetNDauCharged
    10.0,     # Muon_jetPtRelv2
    0.2,      # Muon_jetBTagDeepFlavB
    0.8,      # Muon_jetPtRatio
    1.5,      # Muon_sip3d
    -5.0,     # Muon_log_dxy
    -5.0,     # Muon_log_dz
    0.9,      # Muon_segmentComp
], dtype=np.float64)

In [36]:
python_score = python_evaluator.evaluate(test_event)

print("Python evaluator score:")
print(python_score)
# check that it is in the correct range:
assert np.isfinite(python_score)
assert -1.0 <= python_score <= 1.0

print("Single-event evaluation: PASSED, score in correct range from -1 to 1")

Python evaluator score:
-0.5877358932599654
Single-event evaluation: PASSED, score in correct range from -1 to 1


In [39]:
# test on multiple artificial events
rng = np.random.default_rng(12345)

n_events = 1000

test_events = np.empty((n_events, 13), dtype=np.float64)

test_events[:, 0] = rng.uniform(10, 200, n_events)       # pt
test_events[:, 1] = rng.uniform(-2.4, 2.4, n_events)     # eta
test_events[:, 2] = rng.uniform(0, 1, n_events)          # pfRelIso03_all
test_events[:, 3] = rng.uniform(0, 0.4, n_events)        # miniPFRelIso_chg
test_events[:, 4] = rng.uniform(0, 0.4, n_events)        # neutral mini iso
test_events[:, 5] = rng.integers(0, 20, n_events)        # jetNDauCharged
test_events[:, 6] = rng.uniform(0, 50, n_events)         # jetPtRelv2
test_events[:, 7] = rng.uniform(0, 1, n_events)          # btag
test_events[:, 8] = rng.uniform(0.05, 1.5, n_events)     # jetPtRatio
test_events[:, 9] = rng.uniform(0, 8, n_events)          # sip3d
test_events[:, 10] = rng.uniform(-20, -3, n_events)      # log dxy
test_events[:, 11] = rng.uniform(-20, -2, n_events)      # log dz
test_events[:, 12] = rng.uniform(0.3, 1, n_events)       # segmentComp

In [40]:
array_scores = python_evaluator.evaluate_array(test_events)

print("Shape:", array_scores.shape)
print("First 10 scores:")
print(array_scores[:10])

Shape: (1000,)
First 10 scores:
[-0.77667767 -0.60109556 -0.9736857  -0.95069724 -0.9350232  -0.24949798
 -0.8170201  -0.9195044  -0.9837363  -0.80987287]


In [43]:
# test that evaluate and evaluate_array do the same thing
single_scores = np.array([
    python_evaluator.evaluate(event)
    for event in test_events
])
difference = array_scores - single_scores

print("Maximum absolute difference:")
print(np.max(np.abs(difference)))

Maximum absolute difference:
2.9747813989899896e-08


In [44]:
assert np.allclose(
    array_scores,
    single_scores,
    rtol=0,
    atol=1e-6,
)

print("evaluate_array() consistency: PASSED")

evaluate_array() consistency: PASSED


In [ ]:
# now switch to SWAN to be able to use ROOT